In [ ]:
import sys
sys.path.insert(0, '../../stock_factor_lab_2025/')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 美股實驗

## 回測期間設定

In [ ]:
START_DATE = '2003-3-31'
END_DATE = '2024-12-31'

## 前置作業

### import

In [ ]:
import talib
from get_data import Data
import backtest
from combinations import sim_conditions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# import plotly.graph_objs as go
import plotly.express as px
from itertools import cycle
from plotly.subplots import make_subplots
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft JhengHei']
# plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
# plt.rcParams['axes.unicode_minus'] = False  # 用來正常顯示負號
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FixedLocator, FixedFormatter
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import itertools
import re

from dataframe import CustomDataFrame

### get data

In [ ]:
data=Data(market='US')

## 資料下載

### 調整後股價

In [ ]:
close = data.get('price:close')

In [ ]:
close

### 未調整股價 (因為資料庫裡沒有所以是FMP下載成csv後轉CustomDataframe，計算每日本益比用)

In [ ]:
non_adj_close = pd.read_csv("../2024_code/stock_price_NonAdj_2003-01-01_2024-12-31.csv", encoding="utf-8").pivot(index='date', columns='symbol', values='close')

In [ ]:
non_adj_close.index = pd.to_datetime(non_adj_close.index)
non_adj_close = CustomDataFrame(non_adj_close.ffill())
non_adj_close

### 盈餘再投資率

- 有關N年加總的指標計算在美股都是用N年的相同月份加總，以2025/1月往前推4年加總為例就是2025 1月+2024 1月+2023 1月+2021 1月
- 因為美股不同公司的會計年度(季度)都不一樣，不像台股可以所有公司都在同一年度或是月份看

In [ ]:
netIncome = data.get('annual_report_fundamentals:netIncome')

# 4 年加總
netIncome_df = netIncome.copy()                     
# 提取index的月份
netIncome_df['month'] = netIncome_df.index.month
# 依據月份分組，對每個月份的每四年進行加總
netprofit_rol = netIncome_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(4, min_periods=4).sum())

# 去除稅後淨利為負 (計算盈再率)
adj_netprofit = netprofit_rol[(netprofit_rol > 0) & (netIncome > 0)]

# 長期投資
longTermInvestments = data.get('annual_report_fundamentals:longTermInvestments')
# 固定資產
propertyPlantEquipmentNet = data.get('annual_report_fundamentals:propertyPlantEquipmentNet')

capex = longTermInvestments + propertyPlantEquipmentNet
# 以月份為單位，所以要減掉 48 個月前的資料 (第四年 - 第0年)
capex_rol = capex.diff(48)

In [ ]:
rr = capex_rol / adj_netprofit

### 本益比

In [ ]:
# 每季本益比 #
pe = data.get('quarter_report:PE')
pe = pe[pe > 0] # FMP不會過濾掉負值，但是本益比為負時不能算

In [ ]:
# 計算每日本益比 #
eps = data.get('quarter_report:EPS')

# 近四季EPS加總
eps_rol = eps + eps.shift(3) + eps.shift(6) + eps.shift(9)

pe_daily = non_adj_close / eps_rol
pe_daily = pe_daily[pe_daily > 0] 

### 年度ROE、股利支付率(配息率)、稅前淨利、上市時長

In [ ]:
roe = data.get('annual_report:ROE')
dpr = data.get('annual_report_fundamentals:dividendPayoutRatio')
income_bf_tax = data.get('annual_report_fundamentals:incomeBeforeTax')
comp_profile = data.get('company_profile')

#### 上市櫃滿兩年
在滿兩年之前都設為False

In [ ]:
stock_data = {}

for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date = row['ipo_date']
    end_date = listed_date + pd.DateOffset(years=2)
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)
    # 在上市日之前和之後兩年內設置為 False
    series.loc[:end_date] = False
    
    stock_data[stock_code] = series

listed_df = pd.concat(stock_data, axis=1)

## 原始條件

### ROE 5年平均大於15%

In [ ]:
# ROE 5年平均 > 15%
roe_df = roe.copy()
# 提取index的月份
roe_df['month'] = roe_df.index.month
# 依據月份分組，對每個月份的每5年計算平均
roe_rol = roe_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(5, min_periods=5).mean())

roe_cond = roe_rol > 0.15

### 盈再率 < 40%

In [ ]:
rr_cond = rr < 0.4

### 稅後淨利或是稅前淨利大於7500萬美元

In [ ]:
netprofit_cond = netIncome > 75000000
income_bf_tax_cond = income_bf_tax > 75000000

### 股利支付率 (配息率) 三年至少40%

In [ ]:
payout_ratio = dpr[(netIncome > 0) & (dpr > 0)]

# 3 年至少 > 40%
payout_df = payout_ratio.copy()
# 提取index的月份
payout_df['month'] = payout_df.index.month
# 依據月份分組
payout_ratio_rol = payout_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(3, min_periods=3).min())

payout_cond = payout_ratio_rol > 0.4

### 上市櫃 (IPO) 滿兩年

In [ ]:
listed_cond = listed_df.resample('M').last()

### 本益比進出場條件

每季本益比

In [ ]:
pe_cond_entry = pe < 12
pe_cond_exit = pe > 30

每月本益比

In [ ]:
daily_pe_entry = (pe_daily < 12).resample('M').last()
daily_pe_exit = (pe_daily > 30).resample('M').last()

---

## 羅素1000清單

In [ ]:
russell_1000_df = pd.read_csv('./russell_component_lists/russell_1000_company.csv')
russell_1000_symbol = russell_1000_df['Symbol'].to_list()
filtered_russell_1000_symbol = [symbol for symbol in russell_1000_symbol if symbol in close.columns]

---

## 回測

- 原始策略分有無每季或每月本益比進出場條件、過濾條件用稅前或稅後淨利(美股才有分，台股一律用稅後淨利)
- 美股會分稅前或稅後淨利是因為波克夏股東信原文用稅前淨利來判斷公司規模，但洪瑞泰(2004)提出來的用稅後淨利篩選，只有前面的實驗有看用稅前淨利的策略，後面都是用稅後淨利，因為差不了多少

In [ ]:
# 無本益比進出場 #
# 稅後淨利條件
orig_all_cond = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)[START_DATE:END_DATE]
# 稅前淨利條件
orig_all_cond_inc_bf_tax = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)[START_DATE:END_DATE]

# 有本益比進出場 #
# 每季本益比 #
# 稅後淨利條件
orig_all_cond_and_pe = ((orig_all_cond & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_cond_exit[START_DATE:END_DATE]))
# 稅前淨利條件
orig_all_cond_and_pe_inc_bf_tax = ((orig_all_cond_inc_bf_tax & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | pe_cond_exit[START_DATE:END_DATE]))

# 每月月底本益比 #
# 稅後淨利條件
orig_all_cond_and_pe_daily = ((orig_all_cond & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE]))
# 稅前淨利條件
orig_all_cond_and_pe_daily_inc_bf_tax = ((orig_all_cond_inc_bf_tax & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | daily_pe_exit[START_DATE:END_DATE]))

稅後淨利、每月月底本益比進出場、每月換股

In [ ]:
orig_all_cond_and_pe_daily_rep = backtest.sim(orig_all_cond_and_pe_daily, resample='M', data=data)
orig_all_cond_and_pe_daily_rep.display()

In [ ]:
rep_all_cond_dic = {}

rep_all_cond_dic['美股_原始條件_稅後淨利_不含本益比進出場'] = orig_all_cond
rep_all_cond_dic['美股_原始條件_稅前淨利_不含本益比進出場'] = orig_all_cond_inc_bf_tax

rep_all_cond_dic['美股_原始條件_稅後淨利_每季本益比進出場'] = orig_all_cond_and_pe
rep_all_cond_dic['美股_原始條件_稅前淨利_每季本益比進出場'] = orig_all_cond_and_pe_inc_bf_tax

rep_all_cond_dic['美股_原始條件_稅後淨利_每月月底本益比進出場'] = orig_all_cond_and_pe_daily
rep_all_cond_dic['美股_原始條件_稅前淨利_每月月底本益比進出場'] = orig_all_cond_and_pe_daily_inc_bf_tax

In [ ]:
rep_all_cond = sim_conditions(rep_all_cond_dic, resample='M', data=data)

In [ ]:
# rep_all_cond.plot_creturns()

In [ ]:
# rep_all_cond.plot_stats()

In [ ]:
rep_all_cond.selected_stock_count_analysis()

In [ ]:
fig = rep_all_cond.reports['美股_原始條件_稅後淨利_不含本益比進出場'].create_stacked_returns_plot(5)
fig.write_image('./img/圖 9美股不含進出場條件策略的個股年獲利貢獻前五高占年報酬比例.svg')

In [ ]:
fig = rep_all_cond.reports['美股_原始條件_稅後淨利_每月月底本益比進出場'].create_stacked_returns_plot(5)
fig.write_image('./img/圖 10 美股使用進出場條件策略的個股年獲利貢獻前五高占年報酬比例.svg')

reports 物件包含滿多可用資訊可以拿來視覺化，如果要加新的報表也可以從reports或是reportcollection去改

In [ ]:
rep_all_cond_df = rep_all_cond.selected_stock_count_analysis()
rep_all_cond_df = rep_all_cond_df.reset_index()

# 分組資料
no_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('不含本益比進出場')]
monthly_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('月底本益比進出場')]

# 提取x軸標籤
no_pe_labels = [s.split('_')[1:3] for s in no_pe['Strategy']]
no_pe_labels = ['_'.join(label) for label in no_pe_labels]

# 設置圖表
fig = plt.figure(figsize=(12, 6))
width = 0.1

# 設置x軸位置
x = range(len(no_pe_labels))

# 繪製bars
bars1 = plt.bar([i - width*1.5 for i in x], no_pe['CAGR (%)'], width, label='CAGR', color='tab:blue', alpha=0.7)
bars2 = plt.bar([i - width/2 for i in x], no_pe['MDD (%)'], width, label='MDD', color='tab:orange', alpha=0.7)
bars3 = plt.bar([i + width/2 for i in x], monthly_pe['CAGR (%)'], width, label='', color='tab:blue', alpha=0.7)
bars4 = plt.bar([i + width*1.5 for i in x], monthly_pe['MDD (%)'], width, label='', color='tab:orange', alpha=0.7)

ax = plt.gca()

# 設置Y軸刻度，以10為間隔
ax.yaxis.set_major_locator(plt.MultipleLocator(10))

# 設置網格線
plt.grid(True, alpha=0.3, which='both', axis='both')

# 設置主要x軸標籤（策略名稱）
ax.set_xticks([i for i in x])
ax.set_xticklabels(no_pe_labels, rotation=0)

# 添加次要x軸標籤（本益比條件）
ax2 = ax.secondary_xaxis('bottom') 
# 調整標籤位置，使其對齊對應的柱狀圖組
ax2.set_xticks([i - width for i in x] + [i + width for i in x])
ax2.set_xticklabels(['不含本益比進出場']*len(no_pe_labels) + ['每月本益比進出場']*len(no_pe_labels), fontsize=10)

# 移動主要x軸到次要x軸的下方
ax.xaxis.set_label_position('bottom')
ax.xaxis.set_ticks_position('bottom')
ax.spines['bottom'].set_position(('outward', 40))

# 設置圖表外觀
ax.tick_params(axis='y', labelsize=10)

plt.ylabel('百分比', fontsize=12)
plt.title('美股 2003-2024 符合所有條件_比較稅前與稅後淨利_有無本益比進出場_每月換股', fontsize=14)
plt.legend(loc='lower center', ncol=2, fontsize=12)
plt.axhline(0, color='red', linewidth=0.5)

# 調整版面配置
plt.subplots_adjust(bottom=0.2)

# 顯示圖表
plt.show()

# fig.savefig('圖7_美股原始策略CAGR與MDD比較圖.svg', dpi=300, bbox_inches='tight')

In [ ]:
# rep_all_cond.plot_reps_stock_counts(['美股_原始條件_稅後淨利_每季本益比進出場', '美股_原始條件_稅後淨利_每月月底本益比進出場'] )

In [ ]:
rep_all_cond.plot_reps_stock_counts()

## 切分時間段2003~2009、2009~2024

In [ ]:
orig_all_cond_2003_2009 = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)[START_DATE:'2009-3-31']
orig_all_cond_bftax_2003_2009 = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)[START_DATE:'2009-3-31']

orig_all_cond_2009_2024 = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)['2009-3-31':END_DATE]
orig_all_cond_bftax_2009_2024 = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)['2009-3-31':END_DATE]

all_cond_and_pe_daily_2003_2009 = ((orig_all_cond_2003_2009 & daily_pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | daily_pe_exit[START_DATE:'2009-3-31']))
all_cond_and_pe_daily_bftax_2003_2009 = ((orig_all_cond_bftax_2003_2009 & daily_pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_bftax_2003_2009) | daily_pe_exit[START_DATE:'2009-3-31']))

all_cond_and_pe_daily_2009_2024 = ((orig_all_cond_2009_2024 & daily_pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | daily_pe_exit['2009-3-31':END_DATE]))
all_cond_and_pe_daily_bftax_2009_2024 = ((orig_all_cond_bftax_2009_2024 & daily_pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_bftax_2009_2024) | daily_pe_exit['2009-3-31':END_DATE]))

In [ ]:
time_period_dic = {}

time_period_dic['美股_稅後淨利_不含本益比進出場_2003-2009'] = orig_all_cond_2003_2009
time_period_dic['美股_稅前淨利_不含本益比進出場_2003-2009'] = orig_all_cond_bftax_2003_2009
time_period_dic['美股_稅後淨利_不含本益比進出場_2009-2024'] = orig_all_cond_2009_2024
time_period_dic['美股_稅前淨利_不含本益比進出場_2009-2024'] = orig_all_cond_bftax_2009_2024

time_period_dic['美股_稅後淨利_每月月底本益比進出場_2003-2009'] = all_cond_and_pe_daily_2003_2009
time_period_dic['美股_稅前淨利_每月月底本益比進出場_2003-2009'] = all_cond_and_pe_daily_bftax_2003_2009
time_period_dic['美股_稅後淨利_每月月底本益比進出場_2009-2024'] = all_cond_and_pe_daily_2009_2024
time_period_dic['美股_稅前淨利_每月月底本益比進出場_2009-2024'] = all_cond_and_pe_daily_bftax_2009_2024

time_period_rep_collec = sim_conditions(time_period_dic, resample='M', data=data)

In [ ]:
time_period_rep_collec.selected_stock_count_analysis()

In [ ]:
time_period_rep_collec.reports['美股_稅後淨利_每月月底本益比進出場_2003-2009'].display()
time_period_rep_collec.reports['美股_稅後淨利_每月月底本益比進出場_2009-2024'].display()

畫圖 Function

In [ ]:
def plot_grouped_bar_chart(df):

    df.reset_index(inplace=True)
    
    # 拆分 df 的 'Strategy' column
    df[['條件', '策略', '時間段']] = df['Strategy'].str.split('_', expand=True).iloc[:, 1:]

    numeric_columns = df.select_dtypes(include='number').columns
    grouped = df.groupby(['條件', '策略', '時間段'])[numeric_columns].mean().reset_index()

    conditions = grouped['條件'].unique()
    strategies = grouped['策略'].unique()

    # 設置 bar 的寬度和位置
    bar_width = 0.2
    index = range(len(conditions) * len(strategies))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))

    # 繪製 CAGR (%) 的 grouped bar chart
    for i, time_period in enumerate(['2003-2009', '2009-2024']):
        data = grouped[grouped['時間段'] == time_period]
        cagr_values = data['CAGR (%)'].values
        bar_positions = [x + i * bar_width for x in index]
        ax1.bar(bar_positions, cagr_values, bar_width, label=f'{time_period}')

    ax1.set_xticks([x + bar_width / 2 for x in index])
    ax1.set_xticklabels([f'{cond}_{strat}' for cond in conditions for strat in strategies], rotation=0, ha='center', fontsize=14)
    ax1.grid(True, alpha=0.3)

    ax1.set_xlabel('Strategy')
    ax1.set_ylabel('CAGR (%)', fontsize=14)
    ax1.yaxis.set_major_locator(FixedLocator(ax1.get_yticks()))
    ax1.yaxis.set_major_formatter(FixedFormatter([f'{int(x)}' for x in ax1.get_yticks()]))
    ax1.set_title('美股 2009-2009、2009-2024 不同時間段 CAGR 比較', fontsize=18)
    ax1.tick_params(axis='y', labelsize=16)

    # 繪製 MDD (%) 的 grouped bar chart
    for i, time_period in enumerate(['2003-2009', '2009-2024']):
        data = grouped[grouped['時間段'] == time_period]
        mdd_values = data['MDD (%)'].values
        bar_positions = [x + i * bar_width for x in index]
        ax2.bar(bar_positions, mdd_values, bar_width, label=f'{time_period}')

    ax2.set_xticks([x + bar_width / 2 for x in index])
    ax2.set_xticklabels([f'{cond}_{strat}' for cond in conditions for strat in strategies], rotation=0, ha='center', fontsize=14)

    ax2.set_xlabel('Strategy')
    ax2.set_ylabel('MDD (%)', fontsize=14)
    ax2.yaxis.set_major_locator(FixedLocator(ax2.get_yticks()))
    ax2.yaxis.set_major_formatter(FixedFormatter([f'{int(x)}' for x in ax2.get_yticks()]))
    ax2.set_title('美股 2009-2009、2009-2024 不同時間段 MDD 比較', fontsize=18)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='y', labelsize=16)

    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=18)

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()

    # save plt to svg
    # fig.savefig('美股_2003-2024_不同時間段_CAGR_MDD.svg', format='svg', bbox_inches='tight')

In [ ]:
time_period_test_df = time_period_rep_collec.selected_stock_count_analysis()
plot_grouped_bar_chart(time_period_test_df)

## 美股_布林通道濾網

In [ ]:
us_benchmark = data.get('ruaindex:close')['2002':'2024'] # 因為是算MA300布林通道，所以採用時間要拉長一點
upperband, middleband, lowerband = talib.BBANDS(us_benchmark.close, timeperiod=300, nbdevup=2.0, nbdevdn=2.0)

因為是用大盤來算布林通道所以沒有用到data.indicator，直接用Talib

### 建立濾網
1. 第一次跌破下通道先記錄下來，等到下一次跌破而且到更低的收盤價賣出
2. 等回到均線之上再買入

In [ ]:
# 創建一個買賣訊號的DataFrame，初始值全部為True
us_bollinger_signal = pd.Series(True, index=us_benchmark.index)

# 記錄第一次跌破的價格
first_break_price = None
# 記錄是否已經回到lower band之上
crossed_above_lower = False
# 持續 False 狀態直到突破均線
in_selling_state = False

# 遍歷所有的日期
for date in us_benchmark.index:
    price = us_benchmark.close[date]
    lower = lowerband[date]
    middle = middleband[date]
    
    # 第一次跌破下通道，紀錄價格
    if price < lower and first_break_price is None:
        first_break_price = price
        crossed_above_lower = False
    
    # 價格回到下通道之上
    elif price > lower:
        crossed_above_lower = True
    
    # 回到下通道之上後，再次跌破下通道且符合價格低於第一次跌破價格(賣出條件)
    # 1. 曾經跌破下通道 2. 再次跌破 3. 價格低於第一次跌破價格
    if price < lower and crossed_above_lower and price < first_break_price:
        
        us_bollinger_signal[date] = False
        in_selling_state = True
    
    # 突破均線則重置狀態
    if price > middle:
        us_bollinger_signal[date] = True
        first_break_price = None
        crossed_above_lower = False
        in_selling_state = False
    
    # 在突破均線之前保持賣出狀態
    if in_selling_state and price <= middle:
        us_bollinger_signal[date] = False

In [ ]:
bolling_filt = orig_all_cond.copy()

# 將濾網 df 的 index 對齊 us_bollinger_signal_1 的 index，並填充為 True
aligned_signal = us_bollinger_signal.reindex(bolling_filt.index, method='ffill', fill_value=True)

In [ ]:
# 將 df 中日期對應的行設置為 aligned_signal 的值
bolling_filt.loc[aligned_signal.index, :] = aligned_signal.values[:, None]

篩羅素1000

In [ ]:
filtered_russell_1000_symbol = [symbol for symbol in filtered_russell_1000_symbol if symbol not in ['ALAB', 'LOAR']]

In [ ]:
# bf_tax_cond = income_bf_tax > 75000000
orig_cond_bft = rr_cond & roe_cond & income_bf_tax_cond & payout_cond & listed_cond
nodpr_bft_cond = rr_cond & roe_cond & income_bf_tax_cond & listed_cond


overall_russell_filt_conds = {}

overall_russell_filt_conds['所有條件_無本益比'] = orig_all_cond[START_DATE:END_DATE]
# overall_conds['所有條件_無本益比_稅前淨利'] = orig_cond_bft[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_羅素1000'] = orig_all_cond[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_羅素1000_布林通道'] = (orig_all_cond[filtered_russell_1000_symbol] & bolling_filt)[START_DATE:END_DATE]
# overall_conds['所有條件_無本益比_稅前淨利_羅素1000'] = orig_cond_bft[START_DATE:END_DATE][filtered_russell_1000_symbol]
overall_russell_filt_conds['所有條件_無本益比_布林通道'] = (orig_all_cond & bolling_filt)[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件'] = (orig_all_cond & (roe > 0.15))[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_布林通道'] = (orig_all_cond & bolling_filt & (roe[START_DATE:END_DATE] > 0.15))[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_羅素1000'] = (orig_all_cond & (roe[START_DATE:END_DATE] > 0.15))[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond & bolling_filt & (roe[START_DATE:END_DATE] > 0.15))[filtered_russell_1000_symbol][START_DATE:END_DATE]

overall_russell_filt_conds['所有條件_有本益比'] = ((orig_all_cond & daily_pe_entry)[START_DATE:END_DATE]).hold_until(((~orig_all_cond) | daily_pe_exit)[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
overall_russell_filt_conds['所有條件_有本益比_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]


# overall_conds['所有條件_有本益比_稅前淨利'] = (orig_cond_bft[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_cond_bft[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_羅素1000'] = orig_all_cond_and_pe_daily[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_有本益比_羅素1000_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))[filtered_russell_1000_symbol] 
# overall_conds['所有條件_有本益比_稅前淨利_羅素1000'] = (orig_cond_bft[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_cond_bft[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])[filtered_russell_1000_symbol]


overall_russell_filt_collecs = sim_conditions(overall_russell_filt_conds, resample='M', data=data)
overall_russell_filt_collecs.selected_stock_count_analysis()

In [ ]:
overall_russell_filt_collecs.reports["所有條件_有本益比_布林通道"].display()

In [ ]:
russell_filt_df = overall_russell_filt_collecs.selected_stock_count_analysis()

畫圖Function

In [ ]:
def plot_grouped_bolling_chart(russell_filt_df):

    df = russell_filt_df.copy()
    df.reset_index(inplace=True)
    
    # 創建新的欄位來標記是否包含布林通道
    df['has_bolling'] = df['Strategy'].str.contains('布林通道')
    
    # 獲取基本策略名稱（布林通道之前的部分）
    def get_base_strategy(strategy):
        if '布林通道' in strategy:
            return strategy.split('_布林通道')[0]
        return strategy
    
    df['base_strategy'] = df['Strategy'].apply(get_base_strategy)
    
    # 定義策略順序
    strategy_order = [
        "所有條件_無本益比",
        "所有條件_有本益比",
        "所有條件_無本益比_ROE出場條件",
        "所有條件_有本益比_ROE出場條件",
        "所有條件_無本益比_羅素1000",
        "所有條件_有本益比_羅素1000",
        "所有條件_無本益比_ROE出場條件_羅素1000",
        "所有條件_有本益比_ROE出場條件_羅素1000"
    ]
    
    # 按照指定順序篩選基本策略
    base_strategies = [s for s in strategy_order if s in df['base_strategy'].unique()]
    n_strategies = len(base_strategies)
    
    metrics = ['CAGR (%)', 'MDD (%)']
    
    # 設置圖形大小和樣式
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(max(16, n_strategies * 2), 8))
    bar_width = 0.35
    
    # 設置每組柱狀圖的位置
    indices = range(n_strategies)
    
    # 繪製分組柱狀圖
    for i, (metric, ax) in enumerate(zip(metrics, [ax1, ax2])):
        # 無布林通道的數據
        no_bolling_values = [
            df[(df['base_strategy'] == strategy) & (~df['has_bolling'])][metric].iloc[0]
            if len(df[(df['base_strategy'] == strategy) & (~df['has_bolling'])]) > 0
            else None
            for strategy in base_strategies
        ]
        
        # 有布林通道的數據
        with_bolling_values = [
            df[(df['base_strategy'] == strategy) & (df['has_bolling'])][metric].iloc[0]
            if len(df[(df['base_strategy'] == strategy) & (df['has_bolling'])]) > 0
            else None
            for strategy in base_strategies
        ]
        
        # 繪製柱狀圖
        ax.bar([x - bar_width/2 for x in indices],
               [v for v in no_bolling_values if v is not None],
               bar_width,
               label='無布林通道',
               alpha=0.8)
        
        ax.bar([x + bar_width/2 for x in indices],
               [v for v in with_bolling_values if v is not None],
               bar_width,
               label='有布林通道',
               alpha=0.8)
        
        ax.set_xlabel('策略')
        ax.set_ylabel('百分比 (%)')
        ax.set_title(f'美股_2003~{END_DATE}_{metric} 比較')
        ax.set_xticks(indices)
        ax.set_xticklabels(base_strategies, rotation=45, ha='right', fontsize=14)
        # ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 添加共同的legend在底部中央
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, 
            loc='center',
            bbox_to_anchor=(0.5, -0.01),  # 調整legend的位置
            ncol=2,  # 將legend排成兩列
            fontsize=14,
            title_fontsize=16)

    plt.tight_layout()
    
    plt.show()

    # fig.savefig('./img/圖 20美股2003年至2009年有無加上濾網策略績效比較圖.svg', format='svg', bbox_inches='tight')


plot_grouped_bolling_chart(russell_filt_df)

In [ ]:
overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_無本益比', '所有條件_無本益比_ROE出場條件', '所有條件_無本益比_羅素1000', '所有條件_無本益比_ROE出場條件_羅素1000', '所有條件_有本益比', '所有條件_有本益比_ROE出場條件', '所有條件_有本益比_羅素1000', '所有條件_有本益比_ROE出場條件_羅素1000'])

In [ ]:
overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_無本益比_布林通道', '所有條件_無本益比_ROE出場條件_布林通道', '所有條件_無本益比_羅素1000_布林通道', '所有條件_無本益比_ROE出場條件_羅素1000_布林通道', '所有條件_有本益比_布林通道', '所有條件_有本益比_ROE出場條件_布林通道', '所有條件_有本益比_羅素1000_布林通道', '所有條件_有本益比_ROE出場條件_羅素1000_布林通道'])

In [ ]:
pe_russell_filt_conds = {}

pe_russell_filt_conds['所有條件_有本益比'] = orig_all_cond_and_pe_daily[START_DATE:END_DATE]
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
pe_russell_filt_conds['所有條件_有本益比_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]

pe_russell_filt_collecs = sim_conditions(pe_russell_filt_conds, resample='M', data=data)

pe_russell_filt_collecs.plot_strategies_cumm_return()

In [ ]:
pe_russell_filt_collecs.plot_strategies_MDD()

---

## 選股策略使用不同篩選標準

### 一次調整一個條件

畫圖Function

In [ ]:
def plot_strategy_comparison(df, compare='CAGR (%)', title=None, order_list=None, pattern=False):
    """
    繪製策略比較的柱狀圖
    
    Parameters:
    -----------
    df : pandas DataFrame
        包含策略資訊的數據框
    compare : str, default='CAGR (%)'
        要比較的指標欄位名稱
    title : str, default=None
        自訂圖表標題，若為None則使用預設標題
    order_list : list, default=None
        指定策略前綴的顯示順序，若為None則使用原始順序
    pattern : bool, default=False
        是否顯示子標籤模式
    """
    
    # 取得所有策略名稱並分組
    strategies = df['Strategy'].tolist()
    strategy_pairs = {}
    
    for strategy in strategies:
        if '有本益比進出場' in strategy:
            prefix = strategy.replace('有本益比進出場', '')
            strategy_type = '有本益比'
        else:
            prefix = strategy.replace('無本益比進出場', '')
            strategy_type = '無本益比'
            
        if prefix not in strategy_pairs:
            strategy_pairs[prefix] = {}
        strategy_pairs[prefix][strategy_type] = df[df['Strategy'] == strategy][compare].values[0]

    # 如果有指定順序，重新排序strategy_pairs
    if order_list is not None:
        ordered_pairs = {k: strategy_pairs[k] for k in order_list if k in strategy_pairs}
        strategy_pairs = ordered_pairs

    # 設定圖表
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 設定柱狀圖位置
    x = range(len(strategy_pairs))
    width = 0.35
    
    # 繪製柱狀圖
    rects1 = ax.bar([i - width/2 for i in x], 
                    [pair['有本益比'] for pair in strategy_pairs.values()],
                    width, label='有本益比')
    
    rects2 = ax.bar([i + width/2 for i in x], 
                    [pair['無本益比'] for pair in strategy_pairs.values()],
                    width, label='無本益比')

    # 設定主要x軸標籤
    ax.set_ylabel(compare)
    ax.set_title(title if title else f'策略比較 - {compare}')
    ax.set_xticks(x)
    ax.set_xticklabels(strategy_pairs.keys(), rotation=45, ha='right', fontsize=14)

    # 處理pattern模式的子標籤
    if pattern:
        # 創建第二個x軸
        ax2 = ax.twiny()
        ax2.spines['top'].set_position(('axes', 1.0))
        
        # 找出所有前綴模式
        prefixes = list(strategy_pairs.keys())
        patterns = set()
        for prefix in prefixes:
            match = re.match(r'([^_]+_[^_]+)_.*', prefix)
            if match:
                patterns.add(match.group(1))
        
        patterns = sorted(list(patterns))
        pattern_positions = {}
        
        # 計算每個模式的平均位置
        for pattern in patterns:
            pattern_indices = [i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]
            if pattern_indices:
                pattern_positions[pattern] = sum(pattern_indices) / len(pattern_indices)
        
        # 設定子標籤
        ax2.set_xlim(ax.get_xlim())
        ax2.set_xticks([pattern_positions[pattern] for pattern in patterns])
        ax2.set_xticklabels(patterns, rotation=0, ha='center', fontsize=14)
        
        # 添加垂直分隔線
        ax.grid(False)  # 關閉默認格線
        
        # 獲取y軸的範圍
        ymin, ymax = ax.get_ylim()
        
        # 在每個pattern的起始和結束位置添加垂直線
        for pattern in patterns:
            pattern_start = min([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) - 0.5
            pattern_end = max([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) + 0.5
            
            # # 添加淺色背景區塊
            # ax.axvspan(pattern_start, pattern_end, 
            #           color='gray', alpha=0.1)
            
            # 添加垂直分隔線
            ax.axvline(x=pattern_start, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
            ax.axvline(x=pattern_end, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
        
    ax.legend(loc='upper right')
    plt.tight_layout()
    
    plt.show()

#### 其他條件固定，ROE參數最佳化

In [ ]:
roe_value_cond = {}
no_roe_conds = (rr_cond & netprofit_cond & payout_cond & listed_cond)[START_DATE:END_DATE]

for i in range(10, 31, 5): # 大於 10~30%
    for n in range(3, 6): # 3, 4, 5年平均
        
        roe_opt_df = roe.copy()
        roe_opt_df['month'] = roe_opt_df.index.month

        roe_df_result = roe_opt_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(n, min_periods=n).mean())

        roe_cond_opt = (roe_df_result > (i/100))[START_DATE:END_DATE]

        roe_value_cond[f'roe_{n}y_{i}_無本益比進出場'] = (roe_cond_opt & no_roe_conds)[START_DATE:END_DATE]
        roe_value_cond[f'roe_{n}y_{i}_有本益比進出場'] = ((roe_cond_opt & no_roe_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(roe_cond_opt & no_roe_conds)) | daily_pe_exit[START_DATE:END_DATE]))
        
roe_collection = sim_conditions(roe_value_cond, resample='M', data=data)
roe_collection.selected_stock_count_analysis()

In [ ]:
roe_collection_df = roe_collection.selected_stock_count_analysis()
roe_collection_df.reset_index(inplace=True)

In [ ]:
# 建立顏色映射
colors = {
    '10': 'tab:blue',
    '15': 'orange',
    '20': 'tab:green',
    '25': 'tab:red', 
    '30': 'tab:purple'
}

# 提取年份和ROE閾值
roe_collection_df['Year'] = roe_collection_df['Strategy'].str.extract(r'roe_(\d)y')
roe_collection_df['ROE'] = roe_collection_df['Strategy'].str.extract(r'_(\d+)_')
roe_collection_df['PE'] = roe_collection_df['Strategy'].str.contains('有本益比')

In [ ]:
# 獲取唯一的年份值
years = sorted(roe_collection_df['Year'].unique())

# 設定長條的寬度
bar_width = 0.15

# 計算每個年份組的位置
positions = np.arange(len(years))

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

fig, ax1 = plt.subplots(figsize=(14, 10))

# 儲存所有CAGR值用於設定y軸範圍
roeopt_cagr_values = []

# 儲存有本益比和無本益比的CAGR值
cagr_with_pe = []
cagr_without_pe = []

# 繪製有本益比的長條圖
bars = []  # 儲存長條物件用於之後設定legend
for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
    mask_with_pe = (roe_collection_df['ROE'] == roe_bound) & (roe_collection_df['PE'])
    data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
                    if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
                    for year in years]
    roeopt_cagr_values.extend(data_with_pe)
    cagr_with_pe.extend(data_with_pe)
    
    # 計算長條的位置
    x = positions + (i - 1.5) * bar_width

    # 繪製長條
    bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROE {roe_bound}%', color=colors[roe_bound])
    bars.append(bar)

# # 繪製無本益比的長條圖
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_without_pe = (roe_collection_df['ROE'] == roe_bound) & (~roe_collection_df['PE'])
#     data_without_pe = [roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
#                       if len(roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                       for year in years]
#     roeopt_cagr_values.extend(data_without_pe)
#     cagr_without_pe.extend(data_without_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     ax2.bar(x, data_without_pe, bar_width, color=colors[roe_bound])

# 計算CAGR平均數
avg_cagr_with_pe = sum(cagr_with_pe) / len(cagr_with_pe)
# avg_cagr_without_pe = sum(cagr_without_pe) / len(cagr_without_pe)

# 設定兩個子圖的共同y軸範圍
# ax1.set_ylim(0, 14.5)
# ax2.set_ylim(0, 14.5)

# 設定左方子圖（有本益比）
ax1.set_xticks(positions)
ax1.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=18)
ax1.set_title(f"美股_ROE變化_有本益比進出場策略的CAGR比較", fontsize=20)
ax1.set_xlabel('平均年份', fontsize=18, labelpad=10)
ax1.set_ylabel('CAGR (%)', fontsize=16)
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# # 設定右方子圖（無本益比）
# ax2.set_xticks(positions)
# ax2.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=16)
# ax2.set_title(f"美股_ROE變化_無本益比進出場策略的CAGR比較", fontsize=16) # \nCAGR平均: {avg_cagr_without_pe:.2f}%'
# ax2.set_ylabel('CAGR (%)', fontsize=12)
# ax2.grid(axis='y', linestyle='--', alpha=0.6)

# 調整版面配置
plt.tight_layout()

# 將legend放在整個圖表的最左側
legend = fig.legend(bars, 
                   [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
                   loc='upper left',
                   bbox_to_anchor=(-0.2, 0.95),  # 調整這些數值以微調legend位置
                   fontsize=16,
                   title="平均ROE標準", title_fontsize=18)


# # 在兩圖中間下方添加legend
# legend = fig.legend(bars, [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
#                    loc='center', bbox_to_anchor=(0.5, 0.02),
#                    ncol=5, frameon=False, fontsize=14)

# 調整子圖之間的間距和底部空間
plt.subplots_adjust(bottom=0.1)  # 為legend留出空間

# 顯示圖表
plt.show()

In [ ]:
plot_strategy_comparison(roe_collection_df, 
                         title='美股_ROE變化_有無本益比進出場策略比較_CAGR(%)',
                         order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                         pattern=True)

In [ ]:
plot_strategy_comparison(roe_collection_df,
                            compare='MDD (%)',
                            title='美股_ROE變化_有無本益比進出場策略比較_MDD(%)',
                            order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                            pattern=True)

#### 其他條件固定，PE進場條件參數最佳化

In [ ]:
pe_entry_value_cond = {}

for pe_entry_value in range(8, 13, 2): # 小於 8~12

    pe_entry_opt_df = (pe_daily < pe_entry_value).resample('M').last()[START_DATE:END_DATE]

    pe_entry_value_cond[f'pe小於_{pe_entry_value}進場'] = ((orig_all_cond & pe_entry_opt_df).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE]))

pe_entry_collection = sim_conditions(pe_entry_value_cond, resample='M', data=data)
pe_entry_collection.selected_stock_count_analysis()

In [ ]:
# pe_entry_collection.reports['pe小於_8進場'].display()

---

### 兩兩一組

畫圖Function

In [ ]:
def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None, 
                         benchmark_param1=None, benchmark_param2=None, rep=None):
    """
    繪製策略熱力圖，當df['Min']==0時顯示灰色，並用白色框線標記Benchmark位置
    如果策略的日期不是從2003年開始，也會顯示灰色

    Parameters:
    -----------
    df : pandas DataFrame
        包含 'Strategy' 和比較欄位的數據框
    compare : str, default='CAGR (%)'
        要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
    x_first : bool, default=True
        True: 條件1為X軸，條件2為Y軸
        False: 條件1為Y軸，條件2為X軸
    figsize : tuple, default=(14, 6)
        圖形尺寸
    title : str, optional
        圖表標題，如果不指定則自動生成
    benchmark_param1 : float, optional
        指標1的基準值
    benchmark_param2 : float, optional
        指標2的基準值
    """

    param1_values = []
    param2_values = []
    not_start_2003 = []  # 儲存不是從2003年開始的策略
    
    # 從策略名稱中提取參數
    for strategy in df['Strategy']:
        date_index = rep.reports[strategy].position.index
        # print(date_index)
        
        # 檢查是否從2003年開始
        starts_from_2003 = False
        if len(date_index) > 0:
            first_date_str = str(date_index[0])
            if first_date_str.startswith('2003'):
                starts_from_2003 = True
                
        parts = strategy.split('_')
        # 處理可能帶有%的數值
        param1 = float(parts[1].replace('%', ''))
        param2 = float(parts[3].replace('%', ''))
        param1_values.append(param1)
        param2_values.append(param2)
        not_start_2003.append(not starts_from_2003)  # 記錄非2003開始的策略
    
    df['Param1'] = param1_values
    df['Param2'] = param2_values
    df['Not2003Start'] = not_start_2003  # 將結果添加到DataFrame
    
    condition1_name = df['Strategy'].iloc[0].split('_')[0]
    condition2_name = df['Strategy'].iloc[0].split('_')[2]

    if x_first:
        pivot_table = df.pivot(
            index='Param2',
            columns='Param1',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition1_name} (%)"
        ylabel = f"{condition2_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param1)
            benchmark_y = pivot_table.index.get_loc(benchmark_param2)
    else:
        pivot_table = df.pivot(
            index='Param1',
            columns='Param2',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition2_name} (%)"
        ylabel = f"{condition1_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param2)
            benchmark_y = pivot_table.index.get_loc(benchmark_param1)
    
    plt.figure(figsize=figsize)
    
    mask = (min_pivot == 0)
    
    # 繪製熱力圖
    sns.heatmap(pivot_table,
                annot=True,
                annot_kws={'size': 14},
                fmt='.2f',
                cmap='coolwarm',
                cbar_kws={'label': compare},
                square=True,
                mask=None)
    
    # 在Min==0或不是從2003年開始的位置上覆蓋統一的灰色方塊
    for i in range(len(pivot_table.index)):
        for j in range(len(pivot_table.columns)):
            # 如果Min==0或不是從2003年開始，則繪製灰色方塊
            if mask.iloc[i, j] or (not_2003_pivot.iloc[i, j] == True):
                plt.gca().add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#808080'))
    
    # 如果有指定Benchmark參數，繪製白色框線
    if benchmark_param1 is not None and benchmark_param2 is not None:
        plt.gca().add_patch(plt.Rectangle((benchmark_x, benchmark_y), 1, 1, 
                                        fill=False, edgecolor='white', linewidth=2))
    
    if title is None:
        title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

#### ROE & 盈餘再投資率

In [ ]:
roe_rr_opt_base_conds = payout_cond & netprofit_cond & listed_cond

roe_rr_pe_opt_conds = {}

for roevalue in range(10, 41, 5): # ROE5y平均 10~40%
    for rrvalue in range(0, 81, 10):  # 盈再率 0~80%
        rrvalue_opt = rr.copy() < (rrvalue/100)
        roe_5y_opt = roe_rol.copy() > (roevalue/100)
        
        roe_rr_opt_all_conds = (roe_rr_opt_base_conds & roe_5y_opt & rrvalue_opt)[START_DATE:END_DATE]

        roe_rr_pe_opt_conds[f'ROE5年平均_{roevalue}%_盈再率_{rrvalue}%__本益比進出場'] = (roe_rr_opt_all_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~roe_rr_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])


roe_rr_pe_opt_collecs = sim_conditions(roe_rr_pe_opt_conds, resample='M', data=data)
roe_rr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_rr_pe_opt_df = roe_rr_pe_opt_collecs.selected_stock_count_analysis()
roe_rr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_rr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_盈再率_本益比進出場_CAGR(%)',
                        figsize=(14, 8),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep=roe_rr_pe_opt_collecs)

In [ ]:
roe_rr_pe_opt_collecs.plot_reps_stock_counts(['ROE5年平均_40%_盈再率_20%__本益比進出場', 'ROE5年平均_35%_盈再率_20%__本益比進出場', 'ROE5年平均_25%_盈再率_20%__本益比進出場', 'ROE5年平均_15%_盈再率_40%__本益比進出場'])

In [ ]:
plot_strategy_heatmap(roe_rr_pe_opt_df,
                      compare='MDD (%)',
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_盈再率_本益比進出場_MDD (%)',
                        figsize=(14, 8),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_rr_pe_opt_collecs)

#### ROE & PE 進場條件

In [ ]:
roe_pee_opt_base_conds = rr_cond & payout_cond & netprofit_cond & listed_cond

roe_pee_opt_conds = {}

for roevalue in range(10, 41, 5): # ROE 10~25%
    for peevalue in range(8, 13, 2):  # 本益比

        roe_5y_opt = roe_rol.copy() > (roevalue/100)
        pee_opt = (pe_daily < peevalue).resample('M').last()
        
        roe_pee_opt_all_conds = (roe_pee_opt_base_conds & roe_5y_opt)[START_DATE:END_DATE]

        roe_pee_opt_conds[f'ROE5年平均_{roevalue}%_本益比_{peevalue}__本益比進出場'] = (roe_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~roe_pee_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])


roe_pee_opt_collecs = sim_conditions(roe_pee_opt_conds, resample='M', data=data)
roe_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_pee_opt_collecs.plot_reps_stock_counts(['ROE5年平均_40%_本益比_10__本益比進出場', 'ROE5年平均_40%_本益比_8__本益比進出場', 'ROE5年平均_25%_本益比_10__本益比進出場', 'ROE5年平均_15%_本益比_12__本益比進出場'])

In [ ]:
roe_pee_opt_df = roe_pee_opt_collecs.selected_stock_count_analysis()
roe_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_pee_opt_df,
                        x_first=1, 
                        title='2003~2024 美股_ROE5年平均_本益比_本益比進出場_CAGR(%)',
                        figsize=(12, 5),
                        benchmark_param1=15,
                        benchmark_param2=12,
                        rep = roe_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_pee_opt_df,
                      compare='MDD (%)',
                        x_first=1, 
                        title='2003~2024 美股_ROE5年平均_本益比_本益比進出場_MDD (%)',
                        figsize=(12, 5),
                        benchmark_param1=15,
                        benchmark_param2=12,
                        rep = roe_pee_opt_collecs)